# MLflow + DagsHub — Breast Cancer Classification Lab

This notebook demonstrates an end-to-end classification pipeline with **MLflow** tracking and **DagsHub** remote experiment registry.

**Pipeline:**
1. Connect MLflow to remote **DagsHub** tracking server (`mohnish1234-git/dev_ops_056`)
2. Load & preprocess Breast Cancer classification dataset
3. Train & evaluate multiple classification models (Logistic Regression, Random Forest, Gradient Boosting, XGBoost)
4. Log experiment parameters, classification metrics, and model artifacts to DagsHub
5. Select and register the best model (`Breast_Cancer_Best_Model`)
6. Verify and promote the registered model to **Production**

## 1. Importing Packages

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import os
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)

from xgboost import XGBClassifier

import mlflow
import mlflow.sklearn
import mlflow.xgboost

import dagshub

import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_STATE = 42
TEST_SIZE = 0.2

## 2. DagsHub MLflow Setup

**DagsHub Integration:**
Uses `dagshub.init(..., mlflow=True)` to point MLflow to the remote tracking server under repo owner `mohnish1234-git`.

In [ ]:
dagshub.init(
    repo_owner="mohnish1234-git",
    repo_name="dev_ops_056",
    mlflow=True
)

### (Alternative) Token-based auth

If `dagshub.init` doesn't prompt correctly in your environment, you can set tracking URI manually:
Uncomment and fill in if needed:

In [ ]:
# os.environ["MLFLOW_TRACKING_URI"] = "https://dagshub.com/mohnish1234-git/dev_ops_056.mlflow"
# os.environ["MLFLOW_TRACKING_USERNAME"] = "mohnish1234-git"
# os.environ["MLFLOW_TRACKING_PASSWORD"] = "YOUR_DAGSHUB_TOKEN"
# mlflow.set_tracking_uri(os.environ["MLFLOW_TRACKING_URI"])

In [ ]:
mlflow.set_experiment("Breast Cancer Classification PBLM 1")

## 3. Data Loading and Preprocessing

In [ ]:
bunch = load_breast_cancer(as_frame=True)
raw_df = bunch.frame  # 30 feature columns + 'target'

print("Raw data shape:", raw_df.shape)
print("Target classes:", dict(zip(bunch.target_names, range(len(bunch.target_names)))))
raw_df.head()

### Data Cleaning

In [ ]:
processed_df = raw_df.copy()
processed_df.columns = [c.strip().replace(" ", "_") for c in processed_df.columns]

before = processed_df.shape[0]
processed_df = processed_df.drop_duplicates()
print(f"Dropped {before - processed_df.shape[0]} duplicate rows")

missing = processed_df.isnull().sum().sum()
print(f"Missing values: {missing}")
if missing > 0:
    processed_df = processed_df.fillna(processed_df.median(numeric_only=True))

processed_df["target"] = processed_df["target"].astype(int)
processed_df.describe().T.head()

### Feature Engineering & Data Splitting

In [ ]:
X = processed_df.drop(columns=["target"])
y = processed_df["target"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

scaler = StandardScaler()
X_train_scaled = pd.DataFrame(
    scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index
)
X_test_scaled = pd.DataFrame(
    scaler.transform(X_test), columns=X_test.columns, index=X_test.index
)

print("Train shape:", X_train_scaled.shape)
print("Test shape :", X_test_scaled.shape)
X_train_scaled.head()

## 4. Build Models

In [ ]:
models = [
    (
        "Logistic Regression",
        LogisticRegression(random_state=RANDOM_STATE)
    ),
    (
        "Random Forest",
        RandomForestClassifier(
            n_estimators=200,
            max_depth=6,
            random_state=RANDOM_STATE
        )
    ),
    (
        "Gradient Boosting",
        GradientBoostingClassifier(
            n_estimators=100,
            max_depth=3,
            learning_rate=0.1,
            random_state=RANDOM_STATE
        )
    ),
    (
        "XGBoost",
        XGBClassifier(
            n_estimators=100,
            max_depth=3,
            learning_rate=0.1,
            random_state=RANDOM_STATE,
            eval_metric="logloss"
        )
    )
]

### Classification Metrics Helper

Metrics tracked:
- **Accuracy**
- **Precision**
- **Recall**
- **F1 Score**
- **ROC AUC**

In [ ]:
def evaluate_classification(y_true, y_pred, y_proba):
    acc = accuracy_score(y_true, y_pred)
    prec = precision_score(y_true, y_pred)
    rec = recall_score(y_true, y_pred)
    f1 = f1_score(y_true, y_pred)
    roc = roc_auc_score(y_true, y_proba)
    return {
        "Accuracy": acc,
        "Precision": prec,
        "Recall": rec,
        "F1": f1,
        "ROC_AUC": roc
    }

In [ ]:
reports = []
trained_models = []

for model_name, model in models:
    model.fit(X_train_scaled, y_train)
    predictions = model.predict(X_test_scaled)
    if hasattr(model, "predict_proba"):
        probas = model.predict_proba(X_test_scaled)[:, 1]
    else:
        probas = predictions

    report = evaluate_classification(y_test, predictions, probas)
    reports.append(report)
    trained_models.append(model)

    print("=" * 50)
    print(model_name)
    print("=" * 50)
    for metric_name, val in report.items():
        print(f"{metric_name:10s}: {val:.4f}")
    print()

## 5. Log All Experiments to DagsHub

Each model becomes one MLflow **run** logged to your DagsHub repo (`mohnish1234-git/dev_ops_056`), capturing hyperparameters, classification metrics, and model artifacts.

In [ ]:
for i, (model_name, _) in enumerate(models):
    report = reports[i]
    model = trained_models[i]

    with mlflow.start_run(run_name=model_name):

        # Params
        mlflow.log_param("Model", model_name)
        mlflow.log_params(model.get_params())

        # Metrics
        for metric_name, metric_val in report.items():
            mlflow.log_metric(metric_name, metric_val)

        # Tags
        mlflow.set_tag("Task", "Classification")
        mlflow.set_tag("Dataset", "sklearn Breast Cancer")

        # Model artifact
        if "XGBoost" in model_name:
            mlflow.xgboost.log_model(model, artifact_path="model")
        else:
            mlflow.sklearn.log_model(model, artifact_path="model")

print("All classification experiments logged to DagsHub!")

## 6. Select and Register the Best Model

We select the model with the **highest F1 Score** as the champion champion model and register it in DagsHub MLflow Model Registry.

In [ ]:
best_index = int(np.argmax([r["F1"] for r in reports]))

best_model_name = models[best_index][0]
best_model = trained_models[best_index]
best_report = reports[best_index]

print("Best Model:", best_model_name)
for k, v in best_report.items():
    print(f"{k:10s}: {v:.4f}")

In [ ]:
model_registry_name = "Breast_Cancer_Best_Model"

with mlflow.start_run(run_name=f"Champion_{best_model_name}") as run:

    # Params
    mlflow.log_param("Model", best_model_name)
    mlflow.log_param("Selection_Metric", "F1 Score")
    mlflow.log_param("Dataset", "sklearn Breast Cancer")
    mlflow.log_params(best_model.get_params())

    # Metrics
    for k, v in best_report.items():
        mlflow.log_metric(k, v)

    # Tags
    mlflow.set_tag("Model_Type", best_model_name)
    mlflow.set_tag("Stage", "Candidate")
    mlflow.set_tag("Task", "Classification")

    # Register model
    if "XGBoost" in best_model_name:
        model_info = mlflow.xgboost.log_model(
            best_model,
            artifact_path="model",
            registered_model_name=model_registry_name
        )
    else:
        model_info = mlflow.sklearn.log_model(
            best_model,
            artifact_path="model",
            registered_model_name=model_registry_name
        )

    run_id = run.info.run_id
    model_uri = model_info.model_uri

print("Run ID    :", run_id)
print("Model URI :", model_uri)
print("Model Name:", model_registry_name)

## 7. Load the Registered Model and Verify

Load the model back from the DagsHub registry to confirm predictions match.

In [ ]:
from mlflow.tracking import MlflowClient

client = MlflowClient()

latest_version = client.get_latest_versions(model_registry_name)[0]

print("Model Name:", latest_version.name)
print("Version   :", latest_version.version)
print("Stage     :", latest_version.current_stage)

In [ ]:
model_uri = f"models:/{model_registry_name}/{latest_version.version}"

loaded_model = mlflow.pyfunc.load_model(model_uri)
print("Model loaded successfully")

predictions = loaded_model.predict(X_test_scaled)
print("First 10 predictions:", predictions[:10])

In [ ]:
acc = accuracy_score(y_test, predictions)
f1 = f1_score(y_test, predictions)
print("Reloaded model Accuracy:", round(acc, 4))
print("Reloaded model F1 Score:", round(f1, 4))

## 8. Promote the Model to Production

Update description and transition stage to **Production**.

In [ ]:
client.update_model_version(
    name=model_registry_name,
    version=latest_version.version,
    description="""
    Champion model for Breast Cancer binary classification.

    Tested successfully before production deployment.

    Dataset: sklearn Breast Cancer
    Selection Metric: F1 Score
    """
)

In [ ]:
client.transition_model_version_stage(
    name=model_registry_name,
    version=latest_version.version,
    stage="Production"
)

print("Model promoted to Production")

In [ ]:
production_model = mlflow.pyfunc.load_model(
    f"models:/{model_registry_name}/Production"
)

prod_predictions = production_model.predict(X_test_scaled)
print("Production model first 10 predictions:", prod_predictions[:10])

In [ ]:
production_versions = client.get_latest_versions(
    model_registry_name,
    stages=["Production"]
)

for m in production_versions:
    print("Version :", m.version)
    print("Run ID  :", m.run_id)
    print("Stage   :", m.current_stage)

## Done

DagsHub repo now contains:
- **Experiments tab** — one run per model with classification metrics (Accuracy, Precision, Recall, F1, ROC_AUC)
- **Models tab** — `Breast_Cancer_Best_Model` registered, with a version promoted to **Production**

- Experiments page URL : https://dagshub.com/mohnish1234-git/dev_ops_056/experiments
- Registered model page URL : https://dagshub.com/mohnish1234-git/dev_ops_056/models

## How DVC?

| Stage | Notebook section | Equivalent script |
|---|---|---|
| 1. Data Ingestion | Step 1 | `src/data_ingestion.py` |
| 2. Data Preprocessing | Step 2 | `src/data_preprocessing.py` |
| 3. Feature Engineering | Step 3 | `src/feature_engineering.py` |
| 4. Model Building | Step 4 | `src/model_building.py` |
| 5. Model Evaluation | Step 5 | `src/model_evaluation.py` |